In [5]:
import jax 
from jax import lax
from jax import random as jrnd
from jax import tree_util as jtu
from jax import numpy as jnp
from jax.numpy import linalg as jnpla
from jax.scipy import linalg as jspla
from jax.scipy.special import factorial
from jax.experimental.jet import jet

from matplotlib import pyplot as plt

from time import time
from collections.abc import Callable
from typing import Any

from numerics.spectral import *
from numerics.sobol import *
from numerics.rbf import *

jax.config.update('jax_enable_x64', True)

Point generation methods

Ugh polyharmonic splines that I thought would be a way more important part of this project than they are. I still think they might outperform VEGAS+, especially for geospatial integration. 

Mean curvature: 
$H = \frac{1}{d} \frac{(1+|\nabla f|^2)\nabla^2 f - (\nabla f^\top H_f \nabla f)}{(1 + |\nabla f|^2)^{3/2}} = \frac{ab-c}{a^{3/2}}$

$\nabla H = \frac{1}{d} \frac{a^{3/2} (\nabla a b + a \nabla b - \nabla c) - \frac{3}{2} \sqrt{a} \nabla a (ab - c)}{a^3}$

$\frac{\nabla H}{H} = \frac{a(ab - c)}{a (\nabla a b + a \nabla b - \nabla c) - \frac{3}{2} \nabla a (ab - c)}$

$\text{Hess}(H) = \frac{1}{d} \frac{\frac{3}{2} \sqrt{a} \nabla a (\nabla a b + a \nabla b - \nabla c)^\top }{a^6}$

We need $\nabla \log{H} = \frac{\nabla H}{H}$ and $J_{\nabla \log{H}} = J_{\frac{\nabla H}{H}} = \frac{\text{Hess}(H) H - ||\nabla H||||\nabla H||^\top}{H^2}$

SVGD Update:
$x_i^{\ell + 1} = x_i^\ell + \epsilon \hat{\phi}^\star (x_i^{\ell})$

$\hat{\phi}^\star(x) = \frac{1}{n}\sum\limits_{j=1}^n [k(x_j, x) \nabla_{x_j} \log p(x_j) + \nabla_{x_j} k(x_j, x)] = \frac{1}{n}\sum\limits_{j=1}^n [k(x_j, x) \nabla_{x_j} \log p(x_j) + \nabla_{x_j} k(x_j, x)]$

Jacobian of SVGD Update:


In [ ]:
# polyharmonic spline
_eps:float = 1e-14
def _phs(x:jax.Array, c:jax.Array, r:float, k:int):
    """
    Computes the polyharmonic spline value r^k * log(r).
    
    Parameters
    ----------
    x : jax.Array
        Point of shape (d,).
    c : jax.Array
        Center of shape (d,).
    R : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    jax.Array
        Spline value.
    """
    # Easy
    r_safe = jnp.where(jnp.abs(r) < _eps, 1., r)
    val = jnp.where(k % 2 == 0, jnp.log(r_safe), 1.)
    return r ** k * val

def _Hterms_phs(x:jax.Array, c:jax.Array, r: float, k: int):
    """
    Computes linear terms used in the mean curvature calculation for the polyharmonic spline.
    
    Parameters
    ----------
    x : jax.Array
        Point of shape (d,).
    c : jax.Array
        Center of shape (d,).
    R : float
        Precomputed Euclidean distance ||x - c||.
    k : int, optional
        Order of the spline (default is 2).
    
    Returns
    -------
    grad : jax.Array
        Gradients of shape (d, )
    lapl : float
        Laplacian
    a, b :
        Coefficients
    xmc : 
        Signed per-dimension distance
    R_safe : 
        Safe distance value
    Rkm2 : 
        R^(k-2) term
    """
    # Precursors
    d = x.shape[0]
    xmc = x - c
    r_safe = jnp.where(r < _eps, 1., r)

    # Consolidating terms that come up
    coef_a = jnp.where(k % 2 == 0, jnp.log(r_safe), 1.)
    coef_b = k * coef_a + jnp.where(k % 2 == 0, 1., 0.)
    coef_c = (k - 1) * coef_b + jnp.where(k % 2 == 0, k, 0.)
    coef_d = (k - 2) * coef_c + jnp.where(k % 2 == 0, k * (k - 1), 0.)
    coef_e = (k - 3) * coef_d + jnp.where(k % 2 == 0, k * (k - 1) * (k - 2), 0.)

    f = r**k * coef_a
    f_r = r ** (k - 1) * coef_b
    f_rr = r ** (k - 2) * coef_c
    f_rrr = r ** (k - 3) * coef_d
    f_rrrr = r ** (k - 4) * coef_e

    def test(xx, cc):
        rr = jnpla.norm(xx - cc)
        return rr ** k * jnp.where(k % 2 == 0, jnp.log(rr), 1.)

    # Grad term
    grad = f_r * xmc / r
    # Maggrad
    maggrad = f_r
    # Laplacian
    lapl = f_rr + (d - 1) / r * f_r
    # Hessian
    I = jnp.eye(d)
    XMC = xmc[None] * xmc[:, None]
    hess = I * f_r / r_safe + XMC * (f_rr - f_r / r_safe) / r_safe**2
    # grad-Hessian-grad
    gHg = f_r * maggrad**2 / r + (f_rr - f_r / r_safe) * (grad @ xmc)**2 / r_safe**2

    term_a = maggrad**2 + 1
    term_b = lapl
    term_c = gHg

    grad_term_a = 2 * f_r * f_rr * xmc / r
    grad_term_b = (f_rrr + (d - 1) * f_rr / r - (d - 1) * f_r / r) * xmc / r
    grad_term_c = (2 * f_r * f_rr**2 + f_r * f_rrr) * xmc / r

    return term_a, term_b, term_c, grad_term_a, grad_term_b, grad_term_c

def _Hsum_phs(w:jax.Array, d: int, curvterms: tuple):
    all_a, all_b, all_c, all_grad_a, all_grad_b, all_grad_c = curvterms
    
    w_a = w @ all_a
    w_b = w @ all_b
    w_c = w @ all_c
    w_a_3half = w_a**(3/2)
    H = (w_a * w_b - w_c) / (d * w_a_3half)

    w_grad_a = w @ all_grad_a
    w_grad_b = w @ all_grad_b
    w_grad_c = w @ all_grad_c
    grad_H_num1 = w_a * (w_a * w_grad_b + w_b * w_grad_a - w_grad_c)
    grad_H_num2 = (3 / 2) * w_grad_a * (w_a * w_b - w_c)
    grad_H_denom = w_a_3half * w_a
    grad_H = (grad_H_num1 - grad_H_num2) / grad_H_denom

    return H, grad_H

def _gradterms_H_phs(x:jax.Array, c:jax.Array, r:jax.Array, w:jax.Array, k:int):
    d = x.shape[-1]
    curvterms = jax.vmap(_Hterms_phs, in_axes = (None, 0, 0, None))(x, c, r, k)
    H, grad_H = _Hsum_phs(w, d, curvterms)
    return H, grad_H

def _gradterms_gaussian(x:jax.Array, c:jax.Array, r:float, sigma:float):
    # Precompute
    rsq = r**2
    sigmasq = sigma**2
    xmc = x - c

    # Value
    krn = jnp.exp(- rsq / (2 * sigmasq))

    # Grad
    grad_krn = -xmc * krn / sigmasq

    # Hessian (Jacobian of gradient)
    d = x.shape[-1]
    I = jnp.eye(d)
    XMC = jnp.outer(xmc, xmc)
    hess_krn = krn / sigmasq * (XMC / sigmasq - I)
    return krn, grad_krn, hess_krn

#TODO:
# Make eta/eps (gotta figure that out) actually MEAN something. A maximum adjustment size, something like that. 
# Add an edge smoothing parameter for the logit transform s.t. the kernel at the edges will vanish. 
# Add logit transform...
# Maybe calculate the gradient of the mean curvature by hand. 
def _curvupd_phs(i:int, x:jax.Array, c:jax.Array, R:jax.Array, w:jax.Array, k:int, sigma:float, eps:float):
    # Precursors
    n, d = x.shape
    # Take ith entry of x
    xi = jnp.take(x, i, axis = 0)

    # Calculate H, gradient of H, Hessian of H...
    M, grad_M, hess_M = jax.vmap(_gradterms_H_phs, in_axes = (0, None, 0, None, None))(x, c, R, w, k)
    M_safe = jnp.where(jnp.abs(M) < eps, eps, M)

    # Calculate kernel, gradient, and Hessian
    ri = jnpla.norm(xi[None] - x, axis = -1)
    krn, grad_krn, hess_krn = jax.vmap(_gradterms_gaussian, in_axes = (None, 0, 0, None))(xi, x, ri, sigma)

    # Here's the update
    x_upd = xi + eps * (krn @ grad_M + grad_krn.sum())

    # Now we need all of the special values associated with xi...
    Mi, grad_Mi, hess_Mi, hess_krni = jtu.tree_map(lambda x: jnp.take(x, i, axis = 0), (M, grad_M, hess_M, hess_krn))
    # To build the Jacobian!
    I = jnp.eye(d)
    hesslnM_ii_term = (hess_Mi * Mi - jnp.outer(grad_Mi, grad_Mi)) / Mi**2
    gkrnlnM_term = jax.vmap(jnp.outer, in_axes = (0, 0))(grad_krn, grad_M / M_safe[:, None]).sum(axis = 0)
    hess_krn_term = hess_krn.sum(axis = 0) - hess_krni
    J_krnM = (hesslnM_ii_term + gkrnlnM_term + hess_krn_term)
    Jdet_upd = jnpla.det(I + eps / n * J_krnM)

    uval = krn @ grad_M + grad_krn.sum()
    return x_upd, uval, Jdet_upd / n

def test():
    x = jnp.stack(jnp.meshgrid(*[jnp.linspace(-1, 1, 50)]*2), axis = -1).reshape(50**2, 2)
    c = jnp.stack(jnp.meshgrid(*[jnp.linspace(-.5, .5, 3)]*2), axis = -1).reshape(9, 2)
    R = jnpla.norm(x[:, None] - c[None], axis = -1)
    w = jnp.array([1.] + 8 * [0.])
    return x, c, R, w

# x, c, R, w = test()
# dist = 0.05
# idcs = jnp.arange(x.shape[0])
# x_new, upds, J_new = jax.vmap(_curvupd_phs, in_axes = (0, None, None, None, None, None, None, None))(idcs, x, c, R, w, 3, dist, .1)
# fig,ax = plt.subplots()
# qv = ax.quiver(*x.T, *upds.T, jnpla.norm(upds, axis = -1), cmap = 'rainbow')
# ax.set_facecolor('grey')
# plt.colorbar(qv)
# plt.show()
# plt.scatter(*x.T)
# plt.scatter(*x_new.T)

# We have x,xi
# For ONE ITER:
#   Calculate Ri | (n, ni)
#   Calculate [M, dM](x, xi, w) | (ni,), (ni,)
#   Calculate K(xi, xi) | (ni, ni)
#   Calculate dK(xi, xi) | (ni, ni)
#   Write SVGD update MSVGDupd(xi) = xi + eps * [K(xi, xi) @ (dM / M) + dK(xi, xi) @ onevec] / ni
#   Calculate vi = jacdet(MSVGDupd)(xi) using autodiff because fuck that

94.3341178007724
94.33411780077242


In [132]:
def vegas_rbf(f:Callable[[jax.Array], float|jax.Array], g:Callable[[float|jax.Array], float], 
              a:float|jax.Array, b:float|jax.Array, d:int,
              n_iter:int, m_iters:int, k_phs:int = 3, eta:float = 0.005,
              k_poly:int = 3, basis_poly:str = 'M', 
              sampler:str = 'mc', key:jax.Array = jrnd.key(0)):
      #-------------#
      #### SETUP ####
      # Total number of x
      n = n_iter * m_iters
      # Scale to (-1, 1)
      def scale_m11(x):
        return 2 * (x - a) / (b - a) - 1
      # Scale to (a, b)
      def scale_ab(x):
            return (1 + x) * (b - a) / 2 + a
      # Grab sampler & adaptive sampling engine
      _sampler = _sampler_dict[sampler]

      # Take proposal samples...
      x_init = _sampler(a, b, n, d, key)
      # And slice
      x_segs = x_init.reshape(m_iters, n_iter, d)
      x0 = x_segs[0]
      v0 = jnp.ones((n_iter, )) / n_iter

      # Grab polynomial basis
      alpha_poly = jnp.zeros((d,))
      p0 = mv_psi(x0, basis_poly, k_poly, alpha_poly)
      d_p = k_poly ** d
      p_init = jnp.pad(p0, ((0, n - n_iter + d_p), (0, 0)))

      # Evaluate function
      yyy0 = f(x0)
      # Compress and pad
      y0 = g(yyy0)
      y_init = jnp.pad(y0, (0, n - n_iter + d_p))

      #------------------------------------#
      ### FORMING THE BLOCK MATRIX (Phi) ###
      # Calculate distance between for initial points
      R0 = jnpla.norm(x0[:, None] - x0[None], axis = -1)
      # Polyharmonic spline functions which we will use later TODO
      phs_map = jax.vmap(jax.vmap(_phs, in_axes = (None, 0, 0, None)), in_axes = (0, None, 0, None))
      curvupd_phs_map = jax.vmap(_curvupd_phs, in_axes = (0, None, None, None, None, None, None, None))
      # Calculate values of phs TODO
      phs0 = phs_map(x0, x0, R0, k_phs)

      #### TODO: IMPLEMENT THIS INVERSE
      # Form the full block matrix
      n_rem = n - n_iter
      zeros_01 = jnp.zeros((n_iter, n_rem))
      zeros_12 = jnp.zeros((n_rem, d_p))
      eye = jnp.eye(n_rem)
      zeros_22 = jnp.zeros((d_p, d_p))
      Phi0 = jnp.block([[phs0, zeros_01, p0],
                        [zeros_01.T, eye, zeros_12],
                        [p0.T, zeros_12.T, zeros_22]])
      # Form the inverse 
      Phi_inv_init = jnpla.inv(Phi0)
      # Calculate weights
      w_init = Phi_inv_init @ y_init

      #---------------------#
      ### GRADIENT UPDATE ###
      # Collect into state
      init_state = (0, x_init, p_init, y_init, w_init, Phi_inv_init)

      # This is an update inspired by VEGAS-enhanced using 
      #     Stein-Variational Gradient Descent with 
      #     the curvature of the RBF interpolant as the posterior
      def body_fn(state, xi):
            # Initializations
            i, x, p, y, w, Phi_inv = init_state
            i += 1
            j_start = n_iter * i
            j_end = j_start + n_iter

            # Calculate the distance matrix
            Ri = jnpla.norm(xi[:, None] - x[None, :], axis = -1)
            sigma = 0.01
            # Update x and grab weights
            xi, vi = curvupd_phs_map(jnp.arange(xi.shape[0]), xi, x, Ri, w[:-d_p], k_phs, sigma, eta * i)

            # Grab polynomials
            pi = mv_psi(xi, basis_poly, k_poly, alpha_poly)
            # And raw outputs
            yyyi = f(xi)
            # And scalar outputs
            yi = g(yyyi)

            # Update all arrays
            x = lax.dynamic_update_slice_in_dim(x, xi, j_start, axis = 0)
            p = lax.dynamic_update_slice_in_dim(p, pi, j_start, axis = 0)
            y = lax.dynamic_update_slice(y, yi, (j_start,))

            # Masking Phi update
            Ri2 = jnpla.norm(x[:, None, :] - xi[None, :, :], axis = -1)
            phsi = phs_map(x, xi, Ri2, k_phs)
            Phii = jnp.concat([phsi, pi.T], axis = 0)
            ii, jj = jnp.indices(Phii.shape)
            diag_mask = ii - jj == j_start
            Phii = Phii - 0.5 * diag_mask
            unused_mask = jnp.logical_or(ii > j_end, ii < n - d_p)
            Phii = jnp.where(unused_mask, Phii, 0.)

            # Updating Phi, w
            B = jnp.concat([Phii, diag_mask[:, ::-1]], axis = -1)
            C_inv = jnp.flip(jnp.eye(2 * n_iter), axis = 0)
            # Perform the Woodbury Update
            schur = jnpla.inv(C_inv + B.T @ Phi_inv @ B)
            Phi_inv = Phi_inv - Phi_inv @ B @ schur @ B.T @ Phi_inv
            # Update w
            w = Phi_inv @ y

            return state, (xi, yi, yyyi, vi, jnpla.cond(Phi_inv))
            
      # Scan over all samples...
      testiter = body_fn(init_state, x_segs[1])
      (_, x, _, y, Phi_inv, w), (x_allm0, y_allm0, yyy_allm0, v_allm0, conds) = lax.scan(body_fn, init_state, x_segs[1:])
      # And return quadrature points + weights + values
      x = jnp.concat([x0[None], x_allm0])
      y = jnp.concat([y0[None], y_allm0])
      yyy = jnp.concat([yyy0[None], yyy_allm0])
      v = jnp.concat([v0[None], v_allm0])
      return x,v, conds

In [45]:
# NOTE NEEDS (n - n0) % m_iters = 0
engine_dict = {'mc':_pts_mc,
               'rqmc':_pts_rqmc}

def _sched_const(i, m_iters, eps):
    return eps

def _sched_linear(i, m_iters, eps):
    return eps + (i / m_iters)

schedule_dict = {'constant':_sched_const,
                 'linear':_sched_linear}

def vrbfs(f:Callable, a:float|jax.Array, b:float|jax.Array, 
          n0:int, n_iter:int, m_iters:int, n_est:int, d:int, 
          k_phs:int = 4, k_poly:int = 3, basis_poly:str = 'M',
          engine:str = 'qmc', schedule:str = 'linear', 
          schedule_eps:float = 1., key:jax.Array = jrnd.key(0)):
    
    # SETUP
    # -----
    # Grab engine and scheduler
    _engine = engine_dict[engine]
    _scheduler = schedule_dict[schedule]
    # Split keys
    keys = jrnd.split(key, 2)
    # Broadcast bounds
    a, b = jnp.broadcast_to(a, (d,)), jnp.broadcast_to(b, (d,))
    # Calculate total number of points
    n = n0 + m_iters * n_iter
    mn = m_iters * n_iter
    # Grab unscaled x
    x_raw = _engine(a, b, n, d, keys[0])
    # (Scaling)
    def scale(x_raw):
        return 2 * (x_raw - a) / (b - a) - 1
    def unscale(x):
        return (1 + x) * (b - a) / 2 + a
    x = scale(x_raw)

    # POLYNOMIALS
    # -----------
    # Slice x to construct and update block matrix
    x0, xn = x[:n0], x[n0:].reshape(m_iters, n_iter, d)
    # Grab polynomial basis
    zeros_poly = jnp.zeros((d,))
    p0 = mv_psi(x0, basis_poly, k_poly, zeros_poly)
    pd = k_poly ** d
    p = jnp.pad(p0, ((0, n - n0 + pd), (0, 0)))

    # FORMING THE BLOCK MATRIX (Phi)
    # ------------------------------
    # Calculate distance matrix for initial points
    R0 = jnpla.norm(x0[:, None, :] - x0[None, :, :], axis = -1)
    # Calculate polyharmonic spline matrix
    #   (look up "double vmap" if confused)
    phs_map = jax.vmap(jax.vmap(_phs, in_axes = (None, 0, 0, None)), in_axes = (0, None, 0, None))
    phs0 = phs_map(x0, x0, R0, k_phs)
    # Grab zeros for the block matrix.
    # Form the full block matrix
    zeros_01 = jnp.zeros((n0, mn))
    zeros_12 = jnp.zeros((mn, pd))
    eye = jnp.eye(mn)
    zeros_22 = jnp.zeros((pd, pd))
    Phi0 = jnp.block([[phs0, zeros_01, p0],
                      [zeros_01.T, eye, zeros_12],
                      [p0.T, zeros_12.T, zeros_22]])
    # Form the inverse 
    #### TODO: IMPLEMENT THIS INVERSE
    Phi_inv0 = jnpla.inv(Phi0)

    # CALCULATING WEIGHTS
    # -------------------
    # Evaluate the function at these initial points
    y = jax.vmap(f)(unscale(x0))
    # Pad with zeros for unevaluated points and 
    #   polynomials
    y = jnp.pad(y, (0, n - n0 + pd))
    w0 = Phi_inv0 @ y

    # Make gmg map
    gmg_phs_map = jax.vmap(jax.vmap(_gmg_phs, in_axes = (None, 0, 0, None)), in_axes = (0, None, 0, None))

    # Update for a single iteration
    def update(carry, xi):
        i,x,p,y,Phi_inv,w = carry
        # EVALUATE GRADIENT OF MAGNITUDE OF GRADIENT
        #   FOR EXISTING INTERPOLANT
        # ---------------------------
        # Grab start/end indices
        j_start = n0 + i * n_iter
        j_end = j_start + n_iter
        # Get distance matrix between all points
        #   And points for iteration (ones not yet added
        #   will have zero weights so they'll get masked.
        #   This step lets us keep shape homogeneity)
        Ri = jnpla.norm(x[:, None, :] - xi[None, :, :], axis = -1)
        # Grab gmg of interpolant using analytic formula...
        gmgi = gmg_phs_map(x, xi, Ri, k_phs)
        # ...and weights
        gmgi = jnp.einsum('ijk,i->jk', gmgi, w[:-pd])
        # We also normalize the gradient update
        gmgi = gmgi / jnpla.vector_norm(gmgi, axis = 0).sum() * _scheduler(i, m_iters, schedule_eps)

        # UPDATING INTERNALS
        # -----------------
        # Update x (the "where" term is to wrap massive updates back around)
        xi = xi + gmgi
        xi = jnp.where(xi > 1., (xi % 2) - 1, xi)
        xi = jnp.where(xi < -1, (xi % 2) - 1, xi)
        x = lax.dynamic_update_slice_in_dim(x, xi, j_start, axis = 0)
        # Update polynomials 
        pi = mv_psi(xi, basis_poly, k_poly, zeros_poly)
        p = lax.dynamic_update_slice_in_dim(p, pi, j_start, axis = 0)
        # Update y
        yi = jax.vmap(f)(unscale(xi))
        y = lax.dynamic_update_slice(y, yi, (j_start,))
        
        #  CONDITIONING UPDATE FOR PHI
        # -----------------------
        # Build a little PHS matrix for these updated points
        Ri2 = jnpla.norm(x[:, None, :] - xi[None, :, :], axis = -1)
        phsi = phs_map(x, xi, Ri2, k_phs)
        # And attach polynomial evaluations
        Phii = jnp.concat([phsi, pi.T], axis = 0)
        # Grab indices for masking
        ii,jj = jnp.indices(Phii.shape)
        # Set diagonal equal to -0.5 so that
        #   it will cancel the identity when we update
        #   Phi0
        diag_mask = ii - jj == j_start
        Phii = Phii - 0.5 * diag_mask
        # Mask out unnecessary points
        unused_mask = jnp.logical_or(ii < j_end, ii > n - pd)
        Phii = jnp.where(unused_mask, Phii, 0.)

        # UPDATING PHI, w
        # --------------------------------
        #   Our matrix now admits the decomposition 
        #   Phi_stari ⊗ diag_mask + diag_mask ⊗ Phi_stari. 
        #   This is the same as B @ C @ B.T, where
        #   B is a block matrix of our vectors and flipped indices
        #   and C is a reversed identity. 
        #   (Note that C is selv-inverse, so we just call it C_inv!)
        B = jnp.concat([Phii, diag_mask[:, ::-1]], axis = -1)
        C_inv = jnp.flip(jnp.eye(2 * n_iter), axis = 0)
        # Perform the Woodbury Update
        schur = jnpla.inv(C_inv + B.T @ Phi_inv @ B)
        Phi_inv = Phi_inv - Phi_inv @ B @ schur @ B.T @ Phi_inv
        # Update w
        w = Phi_inv @ y
        carry = (i + 1,x,p,y,Phi_inv,w)

        return carry, unscale(x)
    
    # Scanning
    init = (0,x,p,y,Phi_inv0,w0)
    final,xc = lax.scan(update, init, xn)
    _,xf,pf,yf,Phi_invf,wf = final

    # Evaluate the integrand using the interpolant
    x_est = engine_dict[engine](-1., 1., n_est, d)
    p_est = mv_psi(x_est, basis_poly, k_poly, zeros_poly)
    R_est = jnpla.norm(x[:, None, :] - x_est[None, :, :], axis = -1)
    phs_est = phs_map(x, x_est, R_est, k_phs)
    Phi_est = jnp.concat([phs_est, p_est.T], axis = 0)
    y_est = wf @ Phi_est
    return y_est.mean()

In [ ]:
from scipy.integrate import nquad

def f(x):
    x = jnp.array(x)
    term1 = (4 - 2.1*x[0]**2 + x[0]**4/3) * x[0]**2
    term2 = jnp.prod(x)
    term3 = 4 * x[1]**2 * (x[1]**2 - 1)
    return term1 + term2 + term3

def test_methods(f, a, b, n0, n_iter, m_iters, d):
    n = n0 + n_iter * m_iters
    k_phs = 4

    a,b = jnp.broadcast_to(a, (d,)), jnp.broadcast_to(b, (d,))
    ab = [(ai,bi) for (ai,bi) in zip(a,b)]

    f_scp = lambda *x: f(x)
    I = nquad(f_scp, ab)[0]

    # Monte Carlo estimate
    x_mc = _pts_mc(a, b, n, 2, jrnd.key(1))
    y_mc = jax.vmap(f)(x_mc)
    w_mc = jnp.full_like(y_mc, jnp.prod(b - a) / n)
    I_mc = y_mc @ w_mc


    print('True: %s, MC: %s'%(I, I_mc))
    # VRBFS estimate
    for i in range(6):
        I_vrbfs= vrbfs(f, a, b, n0, n_iter, m_iters, n, 2, k_phs, engine = 'qmc', schedule = 'linear', key = jrnd.key(i))
        print(I_vrbfs)

test_methods(f, -jnp.pi, jnp.pi, 20, 25, 4, 2)


True: 3268.6508153907625, MC: 3643.8260483971694


KeyError: 'qmc'